<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [3]</a>'.</span>

In [1]:
# Parameters
kafka_servers = "kafka:9092"
kafka_topic = "crypto_data"
output_path = "/opt/airflow/data/processed/crypto/"
checkpoint_path = "/opt/airflow/data/checkpoints/crypto/"
execution_date = "2025-06-13"
executor_memory = "2g"
driver_memory = "1g"
total_executor_cores = 2
spark_master = "spark://fced0d436c3f:7077"


In [2]:
#Spark Master
#spark_master='spark://41cf611846e0:7077'

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging
import sys
import argparse
from delta import configure_spark_with_delta_pip

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
#Configure Logging
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger(__name__)

In [ ]:
def create_spark_session(spark_master,executor_memory, driver_memory, total_executor_cores):
    spark=SparkSession.builder\
            .appName("CryptoDataTransformation")\
            .master(spark_master)\
            .config("spark.ui.port", "4040")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
            .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
            .config("spark.sql.adaptive.enabled", "true") \
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true")\
            .config("spark.executor.memory", executor_memory) \
            .config("spark.driver.memory", driver_memory) \
            .config("spark.executor.cores", total_executor_cores) \

    return configure_spark_with_delta_pip(spark).getOrCreate()


In [ ]:
def define_crypto_schema():
    '''Define schema for crypto data'''
    return StructType([
        StructField("symbol", StringType(), True),
        StructField("date", StringType(), True),
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", DoubleType(), True),
    ])

In [ ]:
def read_kafka_crypto_stream(spark,kafka_server,kafka_topic):
    """Read crypto data from kafka"""
    return spark\
    .readStream\
    .format("kafka")\
    .option("kafka.bootstrap.servers", kafka_server)\
    .option("subscribe", kafka_topic)\
    .option("startingOffsets", "latest")\
    .option("failOnDataLoss", "false")\
    .load()

In [ ]:
def parse_crypto_json_data(df):
    """Parse JSON crypto data from Kafka"""
    crypto_schema=define_crypto_schema()

    return df.select(col("symbol"),col("date"),col("open"),col("high"),col("low"),col("close"),col("volume"))

In [ ]:
def apply_crypto_transformations(df):
    """Apply business logic transformations to crypto data"""
    return df\
        .withColumn("processed_timestamp",current_timestamp())\
        .withcolumn("processing_date",current_date())\
        .withColumn("volume_rank",dense_rank().over(window.orderBy(col("volume"))))
        

In [ ]:
def apply_crypto_quality_checks(df):
    """Apply  data quality checks to crypto data"""
    return df\
        .withColumn("data_quality_flag",struct(
            when(col("symbol").isNull(), "missing_symbol")
        )).withColumn("data_quality_status",when(col("data_quality_flag").isNull(), "Passed").otherwise("Failed"))

In [ ]:
def write_to_delta_with_partitions(df,output_path,partition_cols=None):
    """Write DataFrame to Delta lake with Partitioning"""
    write=df.write\
        .format("delta")\
        .mode("append")\
        .option("overwriteSchema", "true")
    if partition_cols:
        write=write.partitionBy(*partition_cols)
    
    return write.save(output_path)

In [ ]:
def process_crypto_batch(df,epoch_id,output_path):
    """Process each batch of crypto data"""
    try:
        logger.info(f"Processing crypto batch {epoch_id}")

        if df.coutnt() == 0:
            logger.info(f"No data in batch {epoch_id}, skipping")
            return
        
        #Appply transformations and quality checks
        transformed_df=apply_crypto_transformations(df)
        quality_checked_df=apply_crypto_quality_checks(transformed_df)

        #Seperate data by quality
        clean_data=quality_checked_df.filter(col("data_quality_status") == "Passed")
        failed_data=quality_checked_df.filter(col("data_quality_status") == "Failed")

        #write clean data 
        if clean_data.count()>0:
            write_to_delta_with_partitions(clean_data,output_path["clean_data"],["symbol"])
        
        if failed_data.count()>0:
            write_to_delta_with_partitions(failed_data,output_path["failed_data"],["symbol"])

        logger.info(f"batch {epoch_id} processed: {clean_data.count()} clean , {failed_data.count()} failed")    
    except Exception as e :
        logger.error(f"Error processing crypto batch {epoch_id}: {str(e)}")

In [ ]:
def main():
    parser =argparse.ArgumentParser(description='Transform crypto_data from kafka')
    parser.add_argument('kafka_servers',required=True,help='kafka bootstrap servers')
    parser.add_argument('kafka_topic',required=True,help='kafka topic name')
    parser.add_argument('output_path',required=True,help='output path for processed data')
    parser.add_argument('checkpoint_path',required=True,help='checkpoint path for streaming')
    parser.add_argument('execution_date',required=True,help='Airflow execution date')
    parser.add_argument('executor_memory',default='2g',help='Executor memory')
    parser.add_argument('driver_memory',default='1g',help='Driver memory')
    parser.add_argument('total_executor_cores',default='2',help='Total executor cores')
    parser.add_argument('spark_master',default='',help='Spark master URL')

    args=parser.parse_args()

    #Create Spark Session

    spark=create_spark_session(args.spark_master,args.executor_memory, args.driver_memory, args.total_executor_cores)
    spark.sparkContext.setLogLevel("WARN")

    #define output paths
    output_path={
        "clean_data":f"{args.output_path}/clean_data",
        "failed_data":f"{args.output_path}/failed_data"
    }

    try:
        #Read kafka data stream
        kafka_df=read_kafka_crypto_stream(spark,args.kafka_servers,args.kafka_topic)

        #Parse JSON data
        parsed_df=parse_crypto_json_data(kafka_df)

        #Start streaming qury with foreachBatch
        query=parsed_df.writeStream\
            .foreachBatch(lambda df,epoch_id:process_crypto_batch(df,epoch_id,output_path))\
            .Option("checkpointLocation", args.checkpoint_path)\
            .trigger(processingTime='30 seconds')\
            .start()
        
        logger.info(f"Crypto transformation job started successfully")
        
        query.awaitTermination(timeout=60)

    except Exception as e:
        logger.error(f"Error in crypto transformation job: {str(e)}")
        raise
    finally:
        spark.stop()
        logger.info("Spark session stopped")

if __name__ == "__main__":
        main()